# Exploratory Data Analysis on claims.csv dataset

|**OMOHL**|**DESCRIPTION**|
|----------|-----------------------------------------|
|**ORIGIN**| This dataset was obtained via web-scraping on the page colombiacheck from 2018-10-26 to 2026-07-16|
|**MOTIVATION**| This project exist because of misinformation and polarization related-problems within colombian society, the main focus is on train a transformer-based model such as BETO to train on the dataset and been able to understand the labeling done to replicate in future news the label |
|**OBJECTIVE**|Curate this dataset, removing missing values and make decisions of data preprocessing to pass into a nlp model |
|**HYPOTHESIS**|N/A |
|**LIMITATIONS**|The data obtained only constitus one dataset from one web page, the bias and limitations that colombiacheck has are included within this analysis and corpus. The model to be trained does not verify facts, it only reproduces the jugdment to classify news given an organization |

- **dataset features**

        - URL: url of the article being reviewed - str
        - verdict: label being given after reviewed [Falso, Cuestionable, Verdadero, Verdadero pero] - str
        - claim_reviewed: corpus to evaluate - str
        - rating: label given by the claimreviewed - str
        - pub_date: publication data - date
        - has_claimreview: boolen whether the new has reviewed - boolean


# 1. DATA ACQUISITION & INITIAL INSPECTION

In [ ]:
import matplotlib.pyplot as plt
import missingno as msno
import numpy as np
import pandas as pd
import seaborn as sns

from fake_news_co.paths import CLAIMS_CSV # Export the path to the raw data directory

In [ ]:
# Load the claims dataset

df = pd.read_csv(CLAIMS_CSV) #load de data
df = df.drop(columns=["jsonld_types"]) # Drop the "jsonld_types" column as it is not needed for analysis

col_rename = {
    "claim_reviewed": "corpus",
    "pub_date": "date",
    "has_claimreview": "has_review"
}

df = df.rename(columns=col_rename) # Rename columns for clarity
df["rating"] = df["rating"].replace({
    "Verdadero pero...": "Verdadero pero"} ) # Standardize the "rating" column values for consistency

df["date"] = pd.to_datetime(df["date"], errors="coerce") # Convert the "date" column to datetime format, coercing errors to NaT

shape = df.shape
print(f"The dataset has {shape[0]} rows and {shape[1]} columns.")
print("-----"*20)

print("\nColumns in the dataset:")
print(df.columns)
print("-----"*20)


print("\nData types of each column:")
print(df.dtypes)
print("-----"*20)

duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows in the dataset: {duplicates}")
print("-----"*20)

print(df.info()) # Display a concise summary of the DataFrame, including the number of non-null entries and data types

# 2. MISSING VALUES ANALYSIS

In [ ]:
# Count and percentage
missing = pd.DataFrame({
    'count':   df.isna().sum(),
    'percent': df.isna().mean().mul(100).round(2)
}).sort_values('percent', ascending=False)
print(missing[missing['count'] > 0])

# Visual pattern — which rows and features are missing together?
msno.matrix(df)      # white = missing, black = present
msno.heatmap(df)     # correlation between missingness across features
plt.show()

**Missingness here is structural, not random (MNAR).** The empty `corpus` values are not a data-quality problem: `corpus`, `rating` and `date` go missing *together* and **exactly** when `has_review == False` — i.e. the article carries no ClaimReview markup (older articles and the `Chequeo Múltiple` format). This accounts for 1,815 rows (38.16%).

These rows **must** be dropped — a model cannot learn from an empty claim — but dropping them is a **non-random selection**, not neutral cleaning. It keeps only articles that happen to expose ClaimReview markup, which skews the corpus toward recent articles and toward `Falso`. The next cell quantifies how much this selection costs each class; this bias is recorded in the Data Statement.

In [ ]:
# Selection: keep only rows that have a claim (ClaimReview markup present).
# Required (no model input without a claim) but NON-random, so first quantify
# how the selection reshapes each class before applying it.

before = df["verdict"].value_counts()
kept = df.dropna(subset=["corpus"])["verdict"].value_counts()
retention = pd.DataFrame({
    "before": before,
    "after":  kept.reindex(before.index).fillna(0).astype(int),
})
retention["retention_%"] = (retention["after"] / retention["before"] * 100).round(1)
print("Per-class effect of requiring a non-empty corpus (ClaimReview selection):")
print(retention)
print("-----" * 20)

# Apply the selection.
df = df.dropna(subset=["corpus"]).copy() # Drop rows where "corpus" is NaN

# De-duplicate on the claim text: the same claim in two rows would leak across
# a train/test split and inflate metrics (duplicates verified label-consistent).
n_dups = df["corpus"].duplicated().sum()
df = df.drop_duplicates(subset=["corpus"]).copy()
print(f"Rows after selection: {len(df) + n_dups}  ->  after de-duplicating corpus: {len(df)}  ({n_dups} duplicate claims removed)")

msno.matrix(df, labels=True)     # white = missing, black = present
plt.title("Missing Values After Selecting Rows With a Claim", fontsize=14, fontweight="bold", pad=15)
plt.show()

if df.isna().sum().sum() > 0:
    missing_after = pd.DataFrame({
        'count':   df.isna().sum(),
        'percent': df.isna().mean().mul(100).round(2)
    }).sort_values('percent', ascending=False)
    print(missing_after[missing_after['count'] > 0])
else:
    print("\nNo missing values remain after the selection.")

# 3. UNIVARIATE ANALYSIS

In [ ]:
# Organice and set the index of the DataFrame for time series analysis

df = df.sort_values(by="date", ascending=True) # Sort the DataFrame by the "date" column for chronological analysis
df = df.set_index("date") # Set the "date" column as the index for time series analysis

In [ ]:
# Date range of the dataset
min_date = df.index.min()
max_date = df.index.max()
print(f"\nThe dataset covers a date range from {min_date.date()} to {max_date.date()}.")

fig, ax = plt.subplots(1,2,figsize=(14, 6))
plt.suptitle("Number of Claims Over Time", fontsize=16, fontweight="bold", y=1.05)
sns.countplot(data=df, x=df.index.year, palette="viridis", ax=ax[0], hue=df.index.year, legend=False) # Create a count plot of the number of claims per year
bars = ax[0].patches
for bar in bars:
    height = bar.get_height()
    ax[0].annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8) # Annotate each bar with its height
ax[0].set_title("Number of Claims per Year", fontsize=10, fontweight="bold") # Set the title for the first subplot
ax[0].set_xlabel("")
ax[0].set_ylabel("Number of Claims", fontsize=10, fontweight="bold")
sns.countplot(data=df, x=df.index.month, palette="viridis", ax=ax[1], hue=df.index.month, legend=False) # Create a count plot of the number of claims per month
bars = ax[1].patches
for bar in bars:
    height = bar.get_height()
    ax[1].annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8) # Annotate each bar with its height
ax[1].set_title("Number of Claims per Month", fontsize=10, fontweight="bold") # Set the title for the second subplot
ax[1].set_xlabel("")
ax[1].set_ylabel("")
sns.despine(ax=ax[0]) 
sns.despine(ax=ax[1])
plt.tight_layout() # Adjust the layout to prevent overlap
plt.show()

- Even though the minimum date is from 2018, one can see it only got 6 claims reviewed in 2018 and 3 claims on 2019. 

- Interesting enough the claims quota is lower in January and December compared to others month.

- In 2026 (mid-year currently) only 195 claims reviewed, if we extrapolate the average claims for the remaining months we got an approximate value of 407 claims reviewd; showing the claims quota has decreased since 2025.

In [ ]:
# Visualize the distribution of the "verdict" vs "rating" columns

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
plt.suptitle("Distribution of Verdict and Rating", fontsize=16, fontweight="bold", y=1.05)
sns.countplot(x="verdict", data=df, order=df["verdict"].value_counts().index, ax=ax[0], palette="pastel", hue="verdict")
bars = ax[0].patches
for bar in bars:
    height = bar.get_height()
    ax[0].annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3), textcoords="offset points",
                   ha='center', va='bottom', fontsize=10, fontweight='bold')
sns.countplot(x="rating", data=df, order=df["rating"].value_counts().index, ax=ax[1], palette="pastel", hue="rating")
bars = ax[1].patches
for bar in bars:
    height = bar.get_height()
    ax[1].annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3), textcoords="offset points",
                   ha='center', va='bottom', fontsize=10, fontweight='bold')
ax[0].set_title("Distribution of Verdict")
sns.despine(ax=ax[0])
sns.despine(ax=ax[1])
ax[0].set_xlabel("")
ax[1].set_xlabel("")
ax[1].set_title("Distribution of Rating")
plt.show()

- There are discrepancies between the `verdict` (ColombiaCheck listing) and `rating` (ClaimReview) labels, explored in section 4.

- There is a strong class imbalance toward `Falso`. This makes the metric choice critical: **macro-F1** (equal weight to every class) rather than accuracy.

- The task is framed as **multi-class, single-label** classification with 3 categories: `[Falso, Cuestionable, Verdadero]`.

- `Verdadero pero` is a nuanced label; in this methodology it is merged into `Verdadero`, which also helps by increasing that class's (very small) support.

# 4. BIVARIATE & MULTIVARIATE ANALYSIS

In [ ]:
# Verdadero pero merging

df["rating"] = df["rating"].replace({
    "Verdadero pero": "Verdadero"
})
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x="rating", order=df["rating"].value_counts().index, palette="pastel")
bars = plt.gca().patches
for bar in bars:
    height = bar.get_height()
    plt.gca().annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3), textcoords="offset points",
                       ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.title("Distribution of Ratings After Merging 'Verdadero pero' into 'Verdadero'", fontsize=16, fontweight="bold", pad=20)
plt.xlabel("Rating", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(rotation=45)
sns.despine()
plt.show()

In [ ]:
# Discrepancies between "verdict" and "rating" columns

discrepancies = df[df["verdict"] != df["rating"]]
shape = discrepancies.shape
print(f"\nNumber of discrepancies between 'verdict' and 'rating': {shape[0]}")
print("-----"*20)

print("\nDiscrepancies between 'verdict' and 'rating':")
print(discrepancies[["verdict", "rating"]])
print("-----"*20)

print("\nCorpus of discrepancies between 'verdict' and 'rating':")
print(discrepancies["corpus"])

- The 6 discrepancies were manually reviewed via their URLs. In **all 6**, the URL slug and article confirm the `rating` (e.g. `es-falso-que...`, `falsa-relacion...`, `no-fueron...`); it is `verdict` that is wrong.

- **Root cause:** `verdict` is derived from the listing card text by matching the first known label in a fixed order (`Verdadero` precedes `Falso`), so any headline that contains the word *"verdadero"* is mislabeled. `rating` comes from the structured ClaimReview markup and is complete (0 nulls on the selected corpus).

- **Decision:** use `rating` as the training label.

# 5. TEXT ANALYSIS

## 5.1 Basics

In [ ]:
# Text analysis

df["corpus_length"] = df["corpus"].apply(lambda x: len(str(x).split())) # Calculate the length of each claim in terms of word count
df["corpus_length"].describe(percentiles=[.25, .5, .75, .9, .95, .99]) # Display descriptive statistics for the "corpus_length" column

- Length of one is weird, further investigation needs to be done

In [ ]:
df.loc[df["corpus_length"] == 1, "corpus"]

- Those corpus of lengths == 1 are URL and not text itself, need to be removed in order to avoid problems when training the model.

In [ ]:
# Remove rows with corpus length of 1 for further analysis, as they may not provide meaningful insights

df = df[df["corpus_length"] > 1].copy() # Filter the DataFrame to keep only rows where "corpus_length" is greater than 1

In [ ]:
# Visualize the distribution of corpus lengths

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
plt.suptitle("Distribution of Corpus Lengths", fontsize=16, fontweight="bold", y=1.02)
sns.histplot(data=df, x="corpus_length", binwidth=1, kde=False, color="salmon", ax=ax[0], hue="rating", multiple="stack")
ax[0].set_title("Distribution of Corpus Lengths", fontsize=10, fontweight="bold")
ax[0].set_xlabel("Corpus Length (Number of Words)", fontsize=12)
ax[0].set_ylabel("Frequency", fontsize=12)
sns.despine(ax=ax[0])

sns.kdeplot(data=df, x="corpus_length", fill=True, hue="rating", ax=ax[1])
ax[1].set_title("Kernel Density Estimate of Corpus Lengths by Rating", fontsize=10, fontweight="bold")
ax[1].set_xlabel("Corpus Length (Number of Words)", fontsize=12)
ax[1].set_ylabel("Density", fontsize=12)
sns.despine(ax=ax[1])
plt.show()

mean_corpus, median_corpus, max_corpus, min_corpus = df["corpus_length"].mean(), df["corpus_length"].median(), df["corpus_length"].max(), df["corpus_length"].min()
print(f"\nMean corpus length: {mean_corpus:.2f} words")
print(f"Median corpus length: {median_corpus} words")
print(f"Maximum corpus length: {max_corpus} words")
print(f"Minimum corpus length: {min_corpus} words")

- The corpora has very few words with a min of 3 words and a max of 17 words, with a mean of 10 words per corpus, this problem added with the class imbalance will be a challenge when training the model.

- Shorter corpus (<=5) tends in one direction: Exploring

## 5.2 Shorter Words

In [ ]:
# Shorter corpus 

shorter = df[df["corpus_length"] <=5].copy() # Filter the DataFrame to keep only rows where "corpus_length" is less than or equal to 5

sns.countplot(data=shorter, x="corpus_length", palette="pastel", hue="rating")
bars = plt.gca().patches
for bar in bars:
    height = bar.get_height()
    plt.gca().annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3), textcoords="offset points",
                       ha='center', va='bottom', fontsize=8)
plt.title("Distribution of Shorter Corpus Lengths (<5 words) by Rating", fontsize=16, fontweight="bold", pad=20)
plt.xlabel("Corpus Length (Number of Words)", fontsize=12)
plt.ylabel("Count", fontsize=12)
sns.despine()
plt.show()

- The conclusion given before is partially true:

    - Corpus length 3 -> 5 False to 2 Cuestionable
    - Corpus length 4 -> 30 False to 4 Cuestionable
    - Corpus length 5 -> 50 False, 12 Cuestionable and 4 True

## 5.3 Word2Vec + T-SNE

In [ ]:
# Word2Vec
import nltk
import re
import time

from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec
from sklearn.manifold import TSNE

class LossLogger(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0
        self.cumulative_losses = []
        self.per_epoch_losses = []

    def on_epoch_end(self, model):
        cum_loss = model.get_latest_training_loss()
        self.cumulative_losses.append(cum_loss)
        # per-epoch = current cumulative minus previous cumulative
        if self.epoch == 0:
            per_epoch = cum_loss
        else:
            per_epoch = cum_loss - self.cumulative_losses[self.epoch - 1]
        self.per_epoch_losses.append(per_epoch)
        print(f"Epoch {self.epoch}: cumulative = {cum_loss:.1f}, per-epoch = {per_epoch:.1f}")
        self.epoch += 1

In [ ]:
# Visualize with T-SNE and Word2Vec embeddings

logger_claims = LossLogger()

nltk.download("stopwords"); nltk.download("punkt_tab")

_ACCENTS = str.maketrans("áéíóúüÁÉÍÓÚÜ", "aeiouuAEIOUU")
def strip_accents(text): return text.translate(_ACCENTS)

stopwords_set = {strip_accents(w) for w in nltk.corpus.stopwords.words("spanish")}


def text_processing(corpus, stopwords_set):
    """
    Preprocess the text corpus by tokenizing and lowercasing.

    Args:
        corpus (str): The text corpus to process.

    Returns:
        list: A list of processed tokens.
    
    """
    processed_text = []
    for text in corpus:
        text = strip_accents(str(text).lower())   # per string
        text = re.sub(r"[^\w\s]", "", text)        # quitar puntuación
        tokens = nltk.tokenize.word_tokenize(text, language="spanish")
        tokens = [t for t in tokens
                  if t.isalpha() and t not in stopwords_set and len(t) > 2]
        processed_text.append(" ".join(tokens))
    return processed_text



corpus_raw = df["corpus"].values.astype(str)
corpus_processed = text_processing(corpus_raw, stopwords_set)
tokens_corpus = [nltk.tokenize.word_tokenize(text, language="spanish") for text in corpus_processed]
df["tokens_corpus"] = tokens_corpus

# Train Word2Vec model on the tokenized corpus

time_start = time.time()
w2v_claims = Word2Vec(
    sentences=tokens_corpus,
    vector_size=100,
    min_count=2,
    seed=42,
    sg=1,
    callbacks=[logger_claims],
    compute_loss=True,
    epochs=30,
    workers=1
)
print(f"Word2Vec training completed in {time.time() - time_start:.2f} seconds.")


per_epoch = logger_claims.per_epoch_losses
plt.figure(figsize=(10, 6))
sns.lineplot(x=range(1, len(per_epoch)+1), y=per_epoch)
plt.xlabel("Epoch")
plt.ylabel("Loss por epoch")
plt.title("Evolución de la pérdida por epoch en Word2Vec", fontsize=12, fontweight='bold', pad=10)
sns.despine()
plt.show()





In [ ]:
# T-SNE visualization of the Word2Vec embeddings
words_to_plot = w2v_claims.wv.index_to_key[:50]  # Get the first 50 words from the Word2Vec model
vectors = []
labels = []
words_not_found = []
for word in words_to_plot:
    if word in w2v_claims.wv:
        vectors.append(w2v_claims.wv[word])
        labels.append(word)
    else:
        words_not_found.append(word)
if len(words_not_found) > 0:
    print(f"Palabras no encontradas en el vocabulario: {', '.join(words_not_found)}")
        
vectors = np.array(vectors)
if len(vectors)>1:
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(vectors)-1) if len(vectors) > 1 else 1)
    vectors_2d = tsne.fit_transform(vectors)
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=vectors_2d[:, 0], y=vectors_2d[:, 1])
    for i, label in enumerate(labels):
        plt.text(vectors_2d[i, 0]+0.05, vectors_2d[i, 1]+0.05, label, fontsize=8)
    plt.title(f"Visualización t-SNE de {len(words_to_plot)} palabras clave", fontsize=14, fontweight='bold', pad=10)
    plt.xlabel("Dimensión 1")
    plt.ylabel("Dimensión 2")
    sns.despine()
    plt.show()

In [ ]:
# Visualize the most distinctive terms for each class based on their frequency relative to the global frequency

from collections import Counter

classes = ["Falso", "Cuestionable", "Verdadero"]
global_freq = Counter(t for toks in df["tokens_corpus"] for t in toks)
N = sum(global_freq.values())

def distinctive_terms(tokens_series, n=15, min_count=3):
    """
    Identify distinctive terms in a series of tokenized texts based on their frequency relative to the global frequency.

    Args:
        tokens_series (pd.Series): A series of tokenized texts.
        n (int): Number of top distinctive terms to return.
        min_count (int): Minimum count of a term to be considered.
    
    Returns:
        list: A sorted list of tuples containing distinctive terms and their scores.
    """
    cls = Counter(t for toks in tokens_series for t in toks)
    n_cls = sum(cls.values())
    scored = [(w, (cls[w]/n_cls) / (global_freq[w]/N)) for w in cls if cls[w] >= min_count]
    return sorted(scored, key=lambda x: -x[1])[:n]

for c in classes:
    print(f"{c:12}: {[w for w, _ in distinctive_terms(df.loc[df['rating']==c, 'tokens_corpus'])]}")

freqs = [global_freq[w] for w in labels]
plt.figure(figsize=(10, 6))
sc = plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], c=freqs, cmap="viridis", s=30)
plt.colorbar(sc, label="frecuencia")
plt.title(f"Visualización t-SNE de {len(words_to_plot)} palabras clave", fontsize=14, fontweight='bold', pad=10)
plt.xlabel("Dimensión 1")
plt.ylabel("Dimensión 2")
sns.despine()
for i, label in enumerate(labels):
    plt.text(vectors_2d[i, 0] + 0.5, vectors_2d[i, 1] + 0.5, label, fontsize=8)
plt.show()

In [ ]:
print("\n>>Palabras similares a las palabras clave:<<\n")
for word in words_to_plot:
    if word in w2v_claims.wv:
        print(f"Palabras similares a '{word}': {[w for w,s in w2v_claims.wv.most_similar(word, topn=5)]}")

In [ ]:
# Top 10 most common terms by class

from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle("Top-10 términos más frecuentes por clase", fontsize=15, fontweight="bold")
for ax, c in zip(axes, classes):
    counter = Counter(t for toks in df.loc[df["rating"] == c, "tokens_corpus"] for t in toks)
    words, counts = zip(*counter.most_common(10))
    sns.barplot(x=list(counts), y=list(words), ax=ax, color="steelblue")
    ax.set_title(c, fontweight="bold"); ax.set_xlabel("count"); ax.set_ylabel("")
    sns.despine(ax=ax)
plt.tight_layout(); plt.show()

## 5.4 Subwords Analysis

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
df["bert_len"] = df["corpus"].apply(lambda t: len(tok.encode(t)))
df["bert_len"].describe()

In [ ]:
# Visualize the distribution of bert lengths

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
plt.suptitle("Distribution of Corpus Lengths", fontsize=16, fontweight="bold", y=1.02)
sns.histplot(data=df, x="bert_len", binwidth=1, kde=False, color="salmon", ax=ax[0], hue="rating", multiple="stack")
ax[0].set_title("Distribution of BERT Token Lengths", fontsize=10, fontweight="bold")
ax[0].set_xlabel("BERT Token Length (Number of Tokens)", fontsize=12)
ax[0].set_ylabel("Frequency", fontsize=12)
sns.despine(ax=ax[0])

sns.kdeplot(data=df, x="bert_len", fill=True, hue="rating", ax=ax[1])
ax[1].set_title("Kernel Density Estimate of BERT Token Lengths by Rating", fontsize=10, fontweight="bold")
ax[1].set_xlabel("BERT Token Length (Number of Tokens)", fontsize=12)
ax[1].set_ylabel("Density", fontsize=12)
sns.despine(ax=ax[1])
plt.show()

mean_bert, median_bert, max_bert, min_bert = df["bert_len"].mean(), df["bert_len"].median(), df["bert_len"].max(), df["bert_len"].min()
print(f"\nMean BERT token length: {mean_bert:.2f} tokens")
print(f"Median BERT token length: {median_bert} tokens")
print(f"Maximum BERT token length: {max_bert} tokens")
print(f"Minimum BERT token length: {min_bert} tokens")


Because BETO is a transformers which uses subwords instead of words, the lenght is greater with an average of 15 tokens per corpus.

# 6. LABEL SANITY CHECK

In [ ]:
# Sanity Check for Label Leakage
leakage = ["falso", "verdadero", "cuestionable"]
counts = {}
for word in leakage:
    n = int(df["corpus"].str.lower().str.contains(rf"\b{word}\b").sum())
    counts[word] = n
    print(("⚠️  " if n else "✅  ") + f"'{word}' appears in {n} claims")

print("\nSummary of label leakage check:", counts)

- The objective of this sanity check is to identify in how many corpus the labels used by rating appears, it does not mean a leakage per se, but a useful consideration.

- In this case the label "falso" just appeared 6 times in the entire rows, meaning the training is not affected or biased by this occurrence.

# 7. FINAL STATEMENT

- The dataset was cleaned fpr missing values and length considerations where 2935 records were held (4756 records originally).

- For simple data analysis: 2018-2019 claims were just 9. Furthermore, on December and January the claims quota decreases, this could be because in the Latino culture those months are holidays, parties, etc. This is just correlational not causal inference.

- Initially 4 labels were encounted [Falso, Cuestionable, Verdadero, Verdadero pero]; and after a naive decision documented the True label was unified getting {"Falso": 2245, "Cuestionable": 597, "Verdadero": 93}, this shows a strong class imbalance an possible issues when training the model.

- There were 6 discrepancies between the label given by "ranking" and "verdict", for this scenario manually the news URL was taken to verify and after review the decision was to use "ranking" as label.

- The finding most claims (76%) are labeled as false and just 3% True is a discovery itself, demonstrating the fake news and polarization within the country regarding politic.

- Word2Vec + T-SNE was used to plot the words and show similarity between those words, frequency and analysis on the length of each corpus.

- Top-10 most common words per category was plotted where in each category "petro" meaning the President of Colombia Gustavo Petro was the top-1 in each category. Drawing a inference that most of the news are based on the persona.

- Length subwords using BETO was used showing that per corpus got an average of 15 tokens.

- Finally, labels-word identified in the corpus possible leaking or biasing the model, but just 6 corpus had the word "falso", this amount is not significant; therefore no additional checks are required.

